# NVIDIA Research Lab

Build one coherent two-stage system:

1. A **Research Agent** chooses live web searches and produces an evidence trail.
2. A **Briefing Editor** turns only that verified research into a one-page briefing.

> **Mission:** What are NVIDIA's strongest evidence-backed growth drivers and material business risks over the next 12-24 months, as of today?

This is an educational research exercise. Do not produce a buy/sell recommendation, price target, fabricated financials or personalised financial advice. You own only two pieces of code: the two CRAFT prompt functions. Everything else is setup and inspectable API plumbing.

## 1. Setup

Run the next cells once. The project `.env` should contain a single line shaped like this:

```text
ANTHROPIC_API_KEY=your_workshop_key_here
```

Never paste a real key into a notebook cell or commit `.env`. If the file has no key, the setup asks for one without displaying it.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path
import json
import os

load_dotenv('.env', override=True)
if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic API key: ')

MODEL = os.getenv('WORKSHOP_MODEL', 'claude-sonnet-5')
client = Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
WEB_SEARCH = {
    'type': 'web_search_20250305',
    'name': 'web_search',
    'max_uses': 3,
}
print(f'Ready: {MODEL} | web searches capped at {WEB_SEARCH["max_uses"]}')

## 2. Supplied API helpers

Read these before running them. Notice the request fields: `model`, `max_tokens`, `system`, `messages` and, only for the agent, `tools`. The `pause_turn` branch lets a long server-side search continue.

In [ ]:
def _block_dict(block):
    return block if isinstance(block, dict) else block.model_dump(exclude_none=True)


def _visible(response):
    text_parts, searches, sources = [], [], {}
    for raw in response.content:
        block = _block_dict(raw)
        if block.get('type') == 'text':
            text_parts.append(block.get('text', ''))
            for citation in block.get('citations') or []:
                url = citation.get('url')
                if url:
                    sources[url] = {
                        'title': citation.get('title') or url,
                        'url': url,
                        'date': citation.get('page_age') or 'Date not supplied',
                    }
        elif block.get('type') == 'server_tool_use' and block.get('name') == 'web_search':
            searches.append(block.get('input', {}).get('query', ''))
        elif block.get('type') == 'web_search_tool_result':
            for result in block.get('content') or []:
                if isinstance(result, dict) and result.get('url'):
                    sources[result['url']] = {
                        'title': result.get('title') or result['url'],
                        'url': result['url'],
                        'date': result.get('page_age') or 'Date not supplied',
                    }
    return '\n'.join(text_parts).strip(), searches, list(sources.values())


def _run_live_research(system_prompt, task):
    messages = [{'role': 'user', 'content': task}]
    all_text, all_searches, source_map = [], [], {}
    for _ in range(3):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1800,
            system=system_prompt,
            messages=messages,
            tools=[WEB_SEARCH],
        )
        text, searches, sources = _visible(response)
        if text:
            all_text.append(text)
        all_searches.extend(searches)
        source_map.update({source['url']: source for source in sources})
        if response.stop_reason != 'pause_turn':
            return {
                'text': '\n'.join(all_text),
                'searches': all_searches,
                'sources': list(source_map.values()),
                'stop_reason': response.stop_reason,
                'usage': response.usage.model_dump(exclude_none=True),
                'fallback_used': False,
            }
        messages.append({'role': 'assistant', 'content': response.content})
    fallback = Path('data/nvidia_research_fallback.md').read_text(encoding='utf-8')
    return {'text': fallback, 'searches': all_searches, 'sources': [], 'stop_reason': 'fallback', 'usage': {}, 'fallback_used': True}


def run_research_agent(system_prompt, task):
    try:
        result = _run_live_research(system_prompt, task)
        if not result.get('text', '').strip():
            raise RuntimeError('Live research returned no visible text')
        return result
    except Exception as error:
        fallback = Path('data/nvidia_research_fallback.md').read_text(encoding='utf-8')
        print(f'Live search unavailable ({type(error).__name__}). Loading the dated classroom fallback.')
        return {'text': fallback, 'searches': [], 'sources': [], 'stop_reason': 'fallback', 'usage': {}, 'fallback_used': True}


def run_briefing_editor(system_prompt, research, task):
    prompt = f"TASK\n{task}\n\nVERIFIED RESEARCH\n{research['text']}\n\nSOURCE METADATA\n{json.dumps(research['sources'], indent=2)}"
    response = client.messages.create(
        model=MODEL,
        max_tokens=1300,
        system=system_prompt,
        messages=[{'role': 'user', 'content': prompt}],
    )
    text, _, _ = _visible(response)
    return {'text': text, 'sources': research['sources'], 'stop_reason': response.stop_reason, 'usage': response.usage.model_dump(exclude_none=True), 'fallback_used': research['fallback_used']}


def show_result(result):
    if result.get('fallback_used'):
        print('!!! DATED CLASSROOM FALLBACK - NOT LIVE RESEARCH !!!\n')
    for query in result.get('searches', []):
        print(f'[SEARCH] {query}')
    for source in result.get('sources', []):
        print(f"[SOURCE] {source['title']} | {source['url']} | {source['date']}")
    print(f"\n[FINAL]\n{result['text']}")
    print(f"\n[STOP] {result['stop_reason']}")


def score_output(output, checklist):
    checks = []
    for criterion in checklist:
        passed = input(f'{criterion} [y/n]: ').strip().lower().startswith('y')
        checks.append({'criterion': criterion, 'passed': passed})
    score = sum(item['passed'] for item in checks)
    print(f'Score: {score}/{len(checks)}')
    return {'score': score, 'total': len(checks), 'checks': checks}

## 3. Stage one: Research Agent

Your agent needs enough freedom to choose useful searches, but enough structure that its evidence can be audited. Complete or rewrite every CRAFT line below.

In [ ]:
# YOUR TURN: this is the first of only two functions you own.
def build_research_prompt():
    return """
C - Context: You are researching NVIDIA for an educational, source-audited workshop as of today's date.
R - Request: Find the strongest evidence for three growth drivers and three material business risks over the next 12-24 months.
A - Approach: Search the live web. Prefer current primary sources, cross-check consequential claims, separate company claims from independent evidence, and name gaps.
F - Format and constraints: Give an executive summary, then a claim/evidence/source/date/confidence table, drivers, risks, disagreements or missing evidence, and a linked source ledger. No recommendation, price target, fabricated figures or personalised advice.
T - Test: Every major claim has a source and date; primary evidence is distinguished from commentary; uncertainty is visible; exactly three drivers and three risks are present.
""".strip()

RESEARCH_TASK = "What are NVIDIA's strongest evidence-backed growth drivers and material business risks over the next 12-24 months, as of today?"
research_v1 = run_research_agent(build_research_prompt(), RESEARCH_TASK)
show_result(research_v1)

### Score it, improve one CRAFT component, rerun it

Do not improve everything at once. Pick the weakest visible criterion, edit the matching CRAFT line, and compare the second output with the first.

In [ ]:
RESEARCH_CHECKLIST = [
    'Exactly three growth drivers and three material risks are present',
    'Every major claim has a named source and date',
    'Primary sources are distinguished from company or media commentary',
    'Confidence, disagreement and missing evidence are visible',
    'There is no recommendation, price target or invented financial figure',
]
research_score_v1 = score_output(research_v1['text'], RESEARCH_CHECKLIST)

# Now edit one line in build_research_prompt(), rerun that cell, then run this.
research_v2 = run_research_agent(build_research_prompt(), RESEARCH_TASK)
show_result(research_v2)
research_score_v2 = score_output(research_v2['text'], RESEARCH_CHECKLIST)

## 4. Stage two: Briefing Editor

This stage has no search tool and no decision loop. It is a grounded generation call, not another autonomous agent. Its context is the verified research from stage one.

In [ ]:
# YOUR TURN: this is the second function you own.
def build_briefing_prompt(research):
    return """
C - Context: You are an evidence-disciplined briefing editor. The user message contains verified NVIDIA research and its source metadata.
R - Request: Turn only that material into a concise one-page educational investor briefing.
A - Approach: Preserve claim-to-source links, compress repetition, balance upside and downside, and carry uncertainty forward rather than filling gaps.
F - Format and constraints: Use a neutral watchlist thesis, business snapshot, three growth drivers, three risks, evidence to monitor, uncertainties, and inherited source references. No new facts, recommendation, price target or personalised advice.
T - Test: Every briefing claim can be traced to the supplied research; all required sections fit on one readable page; the tone is neutral; no unsupported certainty appears.
""".strip()

BRIEFING_TASK = 'Create the one-page NVIDIA investor briefing.'
briefing_v1 = run_briefing_editor(build_briefing_prompt(research_v2), research_v2, BRIEFING_TASK)
show_result(briefing_v1)

In [ ]:
BRIEFING_CHECKLIST = [
    'The thesis is neutral and does not recommend buying or selling',
    'Three drivers, three risks and evidence to monitor are present',
    'Every factual claim is traceable to the research source ledger',
    'Uncertainty and missing evidence survived the rewrite',
    'The result is concise enough to function as a one-page briefing',
]
briefing_score_v1 = score_output(briefing_v1['text'], BRIEFING_CHECKLIST)

# Edit one line in build_briefing_prompt(), rerun it, then compare.
briefing_v2 = run_briefing_editor(build_briefing_prompt(research_v2), research_v2, BRIEFING_TASK)
show_result(briefing_v2)
briefing_score_v2 = score_output(briefing_v2['text'], BRIEFING_CHECKLIST)

## 5. Final reflection

Explain these to a partner:

1. Where did the Research Agent choose its next action?
2. Why is the Briefing Editor a grounded generation call rather than an autonomous agent?
3. Which source supports the briefing's most important claim?
4. Which CRAFT change produced the clearest measurable improvement?
5. What should require human approval if this system were allowed to place a trade?

**Pipeline:** question → CRAFT instructions → live search → source trail → research → grounded generation → evaluation → improvement.